```{index} collections
```

(collections-section)=

# Data Collections


```{contents}
:local:
:depth: 2
```


A bookstore handles many groups of values: the quantities sold during a week, the items in a customer's basket, stock quantities indexed by product code, and orders waiting for collection. Each group contains several values, but the program does different work with each one. Finding Tuesday's sales, looking up a product code, and selecting the next order are different questions. The way you organize the values should support the question you need to answer.

A **collection** groups values and provides operations for working with them. Arrays are the first collection type we studied: {doc}`Chapter 7 <../07_arrays/0700_arrays>` introduced fixed-length, indexed sequences. Keeping that chapter separate gave us time to practice indexing, traversal, and multidimensional data. It did not put arrays outside the collection category. This chapter broadens our choices to other ready-made collection types in the .NET standard library. These classes are often called **built-in collections** because they come with .NET, although they are library types used by C#, rather than separate C# language features. Arrays have direct C# language syntax, such as `int[]`; collection classes such as `List<int>` are library types. Both belong in our collection-family comparison.

This section introduces the big ideas before the detailed syntax: what a collection promises, how values can be accessed, and how to choose among collection families. You will see eight useful families and small demonstrations of their behavior. The next two sections develop `List<T>` and `Dictionary<TKey,TValue>` in depth. For the other families, your goal here is to recognize the need they address and predict their basic behavior, not to memorize every method or implement a data structure.


## Collections and Operations

```{index} collection; element; collection operation; data structure
```


An **element** is one item stored in a collection. An array of sales quantities has numbers as its elements; a list of product codes has strings. A dictionary stores entries that associate a key with a value. The surrounding collection gives these items an organization and a set of supported actions.

Several operations recur across collection families:

| Operation | Question it answers | Bookstore example |
| --- | --- | --- |
| Add or insert | How does a new item enter the group? | Add a product code to the current basket. |
| Retrieve | How do you obtain a particular item? | Find the stock quantity associated with a product code. |
| Update | How does stored information change? | Replace a recorded quantity after a stock check. |
| Remove | How does an item leave the group? | Remove a canceled item from a basket. |
| Membership test | Is this item or key present? | Check whether a customer has already used a promotion. |
| Traverse | How do you visit the stored items? | Print every item on an order. |
| Inspect size | How many items are stored? | Count the orders waiting for collection. |

Not every collection offers every operation, and the method names differ. A stack deliberately limits which item you retrieve next. An array lets you replace an element but does not let you add a new position to that same array. The operations and their rules form a collection's **contract**: the behavior a programmer can rely on.

A **data structure** is an organization of stored information that supports operations. A library collection gives you an already implemented data structure through a public interface of methods and properties. You can use a list before you know how its storage grows, just as you can call `Math.Round` before implementing a rounding algorithm. Later chapters examine the implementation choices behind these operations.

Keep the question of behavior separate from the question of storage. A queue promises a first-in, first-out processing rule. That rule does not require you to know whether a particular implementation uses an array or linked nodes. Conversely, the fact that two collections both store strings does not mean that their access rules are interchangeable.


(collections-by-access-pattern)=

## Collection Families

```{index} collection; families; access pattern
```


An **access pattern** describes how a program usually retrieves and processes its data. Do you know an item's position? Do you know a key that identifies it? Do you only need to know whether it is present? Must items be processed in arrival order?

In the type names below, `T` stands for an element type you choose, such as `string` or `int`. `TKey`, `TValue`, and `TPriority` name the corresponding key, value, and priority types. The examples later in this section use `new` to construct a collection and dot notation to call its operations. First, read the table for the organizing idea behind each family; the next discussion explains generic type arguments before you examine the individual examples.

**Table: An introductory map of .NET collection families.** Adapted from Microsoft's collection-selection guidance, with original business examples and an added priority-queue entry.[^collection-guide][^priority-queue] These eight families are a teaching selection, not Microsoft's exhaustive classification. Sorted variants and collections for specialized settings extend this map.

| Family | Representative type | Main access rule | Business use |
| --- | --- | --- | --- |
| Array | `T[]`, such as `int[]` | Retrieve by index; length is fixed. | Store one quantity for each of five business days. |
| List | `List<T>` | Retrieve by index; the sequence can grow or shrink. | Build a customer's basket as items are selected. |
| Linked list | `LinkedList<T>` | Follow linked nodes; insert or remove at a known node. | Insert a processing step beside an already located step. |
| Dictionary | `Dictionary<TKey,TValue>` | Retrieve a value using its unique key. | Look up stock by product code. |
| Set | `HashSet<T>` | Test membership; equal values occur only once. | Track distinct customers who used a promotion. |
| Stack | `Stack<T>` | Retrieve the most recently added item first. | Keep a history of actions to undo. |
| Queue | `Queue<T>` | Retrieve the earliest added item first. | Process service requests in arrival order. |
| Priority queue | `PriorityQueue<TElement,TPriority>` | Retrieve an item with the lowest priority value first. | Process urgent requests before routine requests. |

The same business process can need several collections. A bookstore might use a dictionary for stock lookup, a list for basket contents, and a queue for orders awaiting service. Choosing one collection does not commit the whole application to that family.

Three distinctions help you read the table. **Position** identifies where an element currently sits in a sequence. **Identity** identifies which entity you mean, such as a product code. **Processing order** determines which waiting item is handled next. Confusing these ideas produces fragile programs: a product code should not silently change because an earlier item was removed from a list.


## Generic Collections

```{index} generic type; type parameter; type argument; type safety
```


A collection's **family** tells you how it organizes access. Its **element type** tells you what values it stores. These are separate choices: a list of integers and a list of strings use the same sequence operations, but they store different kinds of values.

A **generic type** is defined using one or more **type parameters**, which are placeholders for types. In documentation, `List<T>` means a list whose element type is represented by `T`. When you write `List<string>`, `string` is the **type argument** you have supplied for that parameter. The library provides the reusable class; you choose its type arguments when declaring the collection.

| Library notation | Example in a program | Meaning of the type arguments |
| --- | --- | --- |
| `List<T>` | `List<string>` | Each element is a string. |
| `HashSet<T>` | `HashSet<int>` | Each distinct element is an integer. |
| `Dictionary<TKey,TValue>` | `Dictionary<string, int>` | Keys are strings; values are integers. |
| `PriorityQueue<TElement,TPriority>` | `PriorityQueue<string, int>` | Elements are strings; priorities are integers. |

Generic collections provide **type safety**: the compiler checks whether supplied values can be used as the declared types. A `List<int>` accepts integers; it will not accept the string `"twelve"`. This catches a mismatch before the program runs. It does not establish that every integer is appropriate: a negative stock quantity is still an integer, so business validation remains your responsibility.


### The Standard Library and Namespaces

```{index} namespace; standard library; System.Collections.Generic
```


A **namespace** groups related type names. Most collection classes demonstrated here belong to `System.Collections.Generic`. The `using System.Collections.Generic;` directive lets you write `List<string>` instead of the full name `System.Collections.Generic.List<string>`. It makes names available; it does not create a collection or download a package.

Creating a collection requires constructing an object, as in `new List<string>()`. The parentheses call a **constructor**, which initializes the new object. You will study constructors and your own class definitions in Chapter 10. For now, you are using classes already supplied by the library, just as earlier examples used library methods.

Some modern C# projects import common namespaces automatically. Every runnable example here includes the imports it needs so you can run it independently. The following diagram shows three collection namespaces you may encounter, rather than a complete inventory of .NET collections.

:::{figure} ../../figures/collection_namespaces.svg
:width: 700

Selected .NET collection namespaces: generic collections, older non-generic collections, and concurrent collections.
:::

Prefer the generic collection classes for new code like the examples in this chapter. Older non-generic types, such as `ArrayList` and `Hashtable`, commonly store values through `object` and do not provide the same element-type checking. Recognizing their names can help you read older programs; learning them is not a separate requirement here.

Other library collections address additional needs. Sorted dictionaries and sorted sets maintain ordering by key or element, rather than merely remembering positions. Concurrent collections support coordinated access from multiple threads. Immutable collections produce updated versions without changing the original collection. These are additional dimensions of collection design, not extra families that you must master in this lesson.


(collection-operation-syntax)=
## Similar Operations, Different Methods

```{index} collection; operation comparison; method call; property; indexer
```


The families share familiar questions: how do we put an item in, inspect it, take it out, or count the items? Their method names express different rules for answering those questions. Connect each name to the operation you already understand before learning its spelling. This is a comparison map; the worked family examples below and Sections 8.2–8.3 develop the details.[^collection-operations]

The general syntax for calling an instance method is:

```text
collection.MethodName(argument1, argument2, ...);
resultType result = collection.MethodName(arguments);
```

These are syntax patterns, not programs to paste unchanged. `collection` stands for an existing collection variable; the dot selects a member. Replace `MethodName` with the actual name and supply the required arguments inside parentheses. A method with no arguments still needs `()`. Use the second pattern only when the method returns a value you want to store; `resultType` must match that value's type. Some methods change the collection, some return information, and some do both.

Properties and indexers have different syntax:

```text
int size = collection.Count;
elementType item = sequence[index];
valueType value = dictionary[key];
```

`Count` is a property, so it has no parentheses. An array uses `Length` instead. Square brackets select an array/list position or a dictionary key. `elementType` and `valueType` stand for the actual stored types, such as `string` or `int`. An index and a key are different ways to identify an item, even though both use brackets.


### Put In, Inspect, and Take Out

```{index} Add; Remove; RemoveAt; Push; Pop; Peek; Enqueue; Dequeue
```


The following table shows calls on variables named for their collection family. `item`, `key`, `value`, `index`, and `priority` represent values already available to the program. These calls are alternatives for different types, not one interchangeable API.

| Family | Put an item in | Inspect or retrieve | Take an item out |
| --- | --- | --- | --- |
| Array, `T[]` | Fill an existing position: `array[index] = item` | `array[index]` | No operation shrinks this same array. |
| List, `List<T>` | `list.Add(item)`; `list.Insert(index, item)` | `list[index]` | `list.Remove(item)`; `list.RemoveAt(index)` |
| Linked list, `LinkedList<T>` | `linked.AddFirst(item)`; `linked.AddLast(item)` | `linked.First.Value` or `linked.Last.Value`, when nonempty | `linked.Remove(item)`; `linked.RemoveFirst()`; `linked.RemoveLast()` |
| Dictionary, `Dictionary<TKey,TValue>` | `dictionary.Add(key, value)` | `dictionary[key]`, when the key exists | `dictionary.Remove(key)` |
| Set, `HashSet<T>` | `set.Add(item)` | `set.Contains(item)` tests membership | `set.Remove(item)` |
| Stack, `Stack<T>` | `stack.Push(item)` | `stack.Peek()` inspects the top | `stack.Pop()` removes **and returns** the top |
| Queue, `Queue<T>` | `queue.Enqueue(item)` | `queue.Peek()` inspects the front | `queue.Dequeue()` removes **and returns** the front |
| Priority queue, `PriorityQueue<TElement,TPriority>` | `priorityQueue.Enqueue(item, priority)` | `priorityQueue.Peek()` inspects the next item | `priorityQueue.Dequeue()` removes **and returns** the next item |

`Add`, `Push`, and `Enqueue` all put an item into a collection, but they express different access policies. `Remove`, `Pop`, and `Dequeue` all remove something, but they do not select it in the same way or return the same kind of result. `Peek` reads the next item without removing it. With the default priority comparer, a lower priority value is selected first; equal priorities do not guarantee arrival order.

For a list, `Remove(item)` removes the first equal item and returns a Boolean indicating success. `RemoveAt(index)` removes the item at a position and returns no value. For an integer list, this distinction matters: `Remove(2)` searches for the **value** 2, whereas `RemoveAt(2)` selects the **third position**. A set's `Add` returns `false` for a duplicate. A dictionary's `Add` rejects a duplicate key, while `dictionary[key] = value` inserts a missing key or replaces its current value. Similar names do not guarantee identical behavior.

An array's fixed length is part of its contract. Replacing a slot is an update, not insertion; setting it to zero or null does not remove the slot. Creating another array or using a resizable list is a different operation.


### Size, Membership, and Safe Access

```{index} Count; Length; Contains; ContainsKey; TryGetValue; TryPop; TryDequeue
```


| Family | How many items? | Is this item or key present? |
| --- | --- | --- |
| Array | `array.Length` | `Array.IndexOf(array, item) >= 0` for a one-dimensional array |
| List | `list.Count` | `list.Contains(item)` |
| Linked list | `linked.Count` | `linked.Contains(item)` |
| Dictionary | `dictionary.Count` | `dictionary.ContainsKey(key)` |
| Set | `set.Count` | `set.Contains(item)` |
| Stack | `stack.Count` | `stack.Contains(item)` |
| Queue | `queue.Count` | `queue.Contains(item)` |
| Priority queue | `priorityQueue.Count` | No direct `Contains` method; searching entries is a separate task. |

A membership test answers whether an item exists, not how quickly the search runs. The family demonstrations explain the different access patterns. `foreach` visits elements across these families, but its traversal order follows each type's contract; enumerating a priority queue is not guaranteed to produce priority order.

Before indexing an array or list, check the valid range. Before reading `linked.First.Value`, check that a first node exists. Before calling `Peek`, `Pop`, or `Dequeue`, consider the empty case. These calls throw when no item is available. The library also offers Boolean-returning alternatives: `TryPeek`, `TryPop` for stacks, and `TryDequeue` for queues. Priority queues offer `TryPeek` and `TryDequeue` with both element and priority outputs. Dictionaries offer `TryGetValue` for a key that might be absent.

Their general pattern is:

```text
if (dictionary.TryGetValue(key, out valueType value))
{
    // Use value only after the lookup succeeds.
}
```

`out` allows the method to supply a value through an argument. Here the Boolean result tells the `if` whether the lookup succeeded. Section 8.3 develops this lookup pattern with runnable examples. You do not need to memorize every `Try` signature now; recognize the shared idea of checking success before using a result.

**Make the connections:** A basket needs to add and remove a chosen item; an undo history needs to remove the most recent action; a service line needs to remove the earliest arrival. Explain why these related needs lead to `Add`/`Remove`, `Push`/`Pop`, and `Enqueue`/`Dequeue`. Then identify which operations change size and which only inspect data. Use the following examples to check your predictions.


## Choosing a Collection

```{index} collection selection; duplicate; order; lookup
```


Start with the work the program must do, rather than the longest list of available methods. For a bookstore promotion, suppose the manager asks for four results: the order of product selections, the current stock for a known code, the distinct customers who participated, and the next request to process.

An ordered list preserves selections, including repeated product codes. A dictionary associates a product code with its stock quantity. A set records distinct participating customers. A queue selects the next request if the policy is arrival order; a priority queue is appropriate if a defined priority rule should determine service. These structures can collaborate in one application because they answer different questions.

Use the following questions to explain a choice:

1. **What is one element?** A product code, an integer quantity, or a request is a different unit of information. Define what an entry represents before choosing a type.
2. **How will you retrieve it?** Distinguish an index, an identifying key, a membership test, and a next-item rule.
3. **Do order and duplicates carry information?** Removing duplicates from a transaction history can lose events. Preserving every duplicate in a distinct-customer report can misstate participation.
4. **Does the size change?** Decide whether fixed positions describe the task or whether adding and removing elements is normal work.
5. **What operations must be safe?** Consider missing keys, empty collections, invalid values, and whether another part of the program shares the object.

Performance matters too, but "fast" needs an operation and a situation. A dictionary is designed for lookup by key; it does not make every kind of search or report automatically fast. A linked list can insert beside a known node without shifting a suffix, but finding that node can take traversal. Later chapters develop these costs systematically. Here, explain the behavioral fit first and avoid claims that one family is always best.

Trees and graphs extend the larger study of data structures. They model hierarchies and networks, rather than adding two more standard generic classes named `Tree<T>` and `Graph<T>` to this table. Chapters 17 and 20 teach those structures and their applications. Arrays are developed in Chapter 7; linked structures in Chapter 14; stacks and queues in Chapter 15; priority queues in Chapter 18; and hashing and sets in Chapter 19.

For this chapter's detailed coding and assignments, concentrate on lists and dictionaries. The broader family map should help you recognize alternatives and ask better questions about the data. You do not need the internal mechanics of a hash table or heap to explain why a lookup table and a waiting line have different access rules.


## Collection Examples

```{index} collections; examples
```


You now have the family map, the idea of a collection contract, and a way to explain a choice. The following examples make those ideas concrete. For each one, identify what is stored and predict how the collection retrieves it before running the code. The detailed list and dictionary lessons follow in Sections 8.2 and 8.3.


The next example retains the explicit type names so you can see both choices. It stores employee names in a list and associates names with training scores in a dictionary. The commented line illustrates a type mismatch; leave it commented when running the demonstration.


In [7]:
using System;
using System.Collections.Generic;

List<string> names = new List<string> { "Ana", "Bo" };
Dictionary<string, int> scores = new Dictionary<string, int>
{
    ["Ana"] = 92,
    ["Bo"] = 85
};

Console.WriteLine(names[0]);
Console.WriteLine(scores["Bo"]);
// scores["Ana"] = "ninety-two"; // Expected compile error if uncommented: string is not int.


Ana
85


### Arrays and Lists

```{index} array; list; index; sequence
```


A **sequence** gives its elements positions. In the arrays and lists used here, positions are indexed from zero. Both can contain repeated values. Their order is the order of the sequence, which is not necessarily sorted order.

An array's length remains fixed after creation, but its elements can change. Fixed length does not mean read-only data. A list manages storage as elements are added or removed. Its `Count` reports the number of elements currently in the list; an array uses `Length`.

Read this example as a comparison of behavior. The weekly array still has five positions after a correction. The basket grows when a second item is added, and it retains the order of those additions.


In [1]:
using System;
using System.Collections.Generic;

int[] dailySales = { 12, 9, 15, 8, 11 };
dailySales[1] = 10;

List<string> basket = new List<string> { "NOTEBOOK" };
basket.Add("PEN");

Console.WriteLine($"Business days: {dailySales.Length}");
Console.WriteLine($"Tuesday sales: {dailySales[1]}");
Console.WriteLine($"Basket items: {basket.Count}");
Console.WriteLine($"First item: {basket[0]}");


Business days: 5
Tuesday sales: 10
Basket items: 2
First item: NOTEBOOK


Use an array when a fixed number of indexed positions expresses the task clearly. Use a list when the sequence changes size during normal work. A list is not automatically the best choice simply because it is more flexible: the fixed five-day structure can make the meaning of the array clear. Likewise, repeatedly replacing arrays just to add another basket item makes ordinary work harder than it needs to be.

**Check Your Understanding:** Does changing Tuesday's quantity require a list? What change to the task would make a list more useful?

```{dropdown} Suggested Answer
No. Replacing an existing array element does not change the array's length. A list becomes more useful if you are collecting an unknown or changing number of daily records, rather than maintaining exactly five positions.
```


### Linked Lists

```{index} linked list; node
```


A **linked list** organizes a sequence as **nodes**, each holding an element and links to neighboring nodes. .NET's `LinkedList<T>` is doubly linked: a node can refer to both its previous and its next node. You do not use `steps[1]` to retrieve its second element. You traverse the sequence or work with a node reference.[^linked-list]

Imagine a sequence of order-processing steps. After locating the packing step, you discover that a gift-check step belongs immediately after it. In the example below, `AddLast` returns a reference to the new node; `AddAfter` uses that reference to choose an insertion point. `var` asks C# to infer the node variable's type from the returned value.


In [2]:
using System;
using System.Collections.Generic;

LinkedList<string> steps = new LinkedList<string>();
var packingStep = steps.AddLast("Pack order");
steps.AddLast("Dispatch order");
steps.AddAfter(packingStep, "Check gift message");

foreach (string step in steps)
{
    Console.WriteLine(step);
}


Pack order
Check gift message
Dispatch order


The result is still an ordered sequence, but it has different access operations from an indexed list. Inserting beside a node you already hold avoids shifting later elements. Finding that node can still require a search, and nodes have storage overhead. A linked list is therefore not a universal replacement for `List<T>`. Chapter 14 develops these tradeoffs and their implementations.


### Dictionaries

```{index} dictionary; key; value; key-value pair
```


A **dictionary** associates a **key**, which identifies an entry, with a **value**, which is the information stored for that key. A stock lookup might use a product code as the key and an integer quantity as the value. Unlike a list index, the code expresses identity rather than position.

Keys are unique within a dictionary: one key cannot identify two separate entries. Values do not have to be unique. Two products may have the same stock quantity without being the same product. Here both codes initially map to `12`, and the program retrieves a quantity by code.


In [3]:
using System;
using System.Collections.Generic;

Dictionary<string, int> stock = new Dictionary<string, int>
{
    ["NOTEBOOK"] = 12,
    ["PEN"] = 12
};

Console.WriteLine($"Product codes: {stock.Count}");
Console.WriteLine($"Notebook stock: {stock["NOTEBOOK"]}");


Product codes: 2
Notebook stock: 12


The expression `stock["NOTEBOOK"]` asks for a value associated with a key. It does not ask for the first, second, or alphabetically earliest entry. A dictionary is not an alphabetically sorted list, and you should not make a report depend on its enumeration order. You also need a policy for unknown codes; Section 8.3 teaches safe lookup and the difference between adding an entry and updating one.


### Sets

```{index} set; membership; uniqueness
```


A **set** stores distinct elements and emphasizes **membership**, meaning whether an element is present. Adding an equal element again does not create another copy. A set therefore answers a different question from a list: "Which customers participated?" rather than "What were all the participation events?"

If a customer uses a promotion twice, a list can record both events, while a set can record that customer's participation once. In this demonstration, `Add` returns `false` for the repeated value because no new element was added. `Contains` asks whether an element is in the set.


In [4]:
using System;
using System.Collections.Generic;

HashSet<string> customers = new HashSet<string>();
customers.Add("C101");
customers.Add("C205");
bool addedAgain = customers.Add("C101");

Console.WriteLine($"Distinct customers: {customers.Count}");
Console.WriteLine($"Added C101 again: {addedAgain}");
Console.WriteLine($"C205 participated: {customers.Contains("C205")}");


Distinct customers: 2
Added C101 again: False
C205 participated: True


A set does not preserve the number of times a value appeared. If you need that number, a dictionary can associate each customer with a count. A `HashSet<T>` also does not promise sorted enumeration. Chapter 19 explains hashing, equality, and set operations; here you only need to recognize the membership and uniqueness contract.

**Check Your Understanding:** A list contains the product codes `PEN`, `PEN`, and `NOTEBOOK`. Would a set be a suitable replacement for the basket? Would it be suitable for the distinct products included in a promotion report?

```{dropdown} Suggested Answer
A set would lose the repeated PEN entry, so it would not preserve a basket that uses repeated entries to represent quantity. It would be suitable for a report asking only which distinct product codes appeared. The choice depends on whether duplicates carry information.
```


(rules-for-processing-work)=

### Processing Order

```{index} processing order; stack; queue; priority queue
```


Sometimes the important question is which item should leave the collection next. A list lets you choose a position, but that freedom puts the responsibility for enforcing the processing rule on your program. A stack or queue expresses a particular rule through its operations.

These structures do not decide an organization's policy for you. If requests should be handled in arrival order, choose a collection that expresses that policy. If urgency should take precedence, make the urgency rule explicit and choose a different structure.


### Stacks and Queues

```{index} last in first out; first in first out; push; pop; enqueue; dequeue
```


A **stack** follows **last in, first out (LIFO)**. `Push` adds an item at the top; `Pop` removes and returns the top item. Think of an undo history: the most recent action should normally be the first action undone. Storing a description of an action is only a demonstration of ordering; a real undo feature also needs a way to reverse that action.

A **queue** follows **first in, first out (FIFO)**. `Enqueue` adds an item at the back; `Dequeue` removes and returns the item at the front. This expresses an arrival-order service line.

Both collections below receive `Order A` followed by `Order B`. Predict which order each returns before running the example.


In [5]:
using System;
using System.Collections.Generic;

Stack<string> recentActions = new Stack<string>();
recentActions.Push("Order A");
recentActions.Push("Order B");

Queue<string> waitingOrders = new Queue<string>();
waitingOrders.Enqueue("Order A");
waitingOrders.Enqueue("Order B");

Console.WriteLine($"Stack returns: {recentActions.Pop()}");
Console.WriteLine($"Queue returns: {waitingOrders.Dequeue()}");
Console.WriteLine($"Actions remaining: {recentActions.Count}");
Console.WriteLine($"Orders remaining: {waitingOrders.Count}");


Stack returns: Order B
Queue returns: Order A
Actions remaining: 1
Orders remaining: 1


Removing an item changes the collection: each now has one item left. `Peek` lets you inspect the next item without removing it. Removing from an empty stack or queue requires a separate check or a suitable `Try...` operation; this example establishes that both are nonempty before removal. Chapter 15 develops safe operations and practical applications.

The FIFO rule is an ordering rule, not a complete definition of fairness. For example, an arrival-order queue does not by itself accommodate an urgent safety issue. You still need to decide whether arrival order expresses the policy the task actually requires.


### Priority Queues

```{index} priority queue; priority
```


A **priority queue** chooses the next item using a priority value rather than its arrival position. In .NET's default `PriorityQueue<TElement,TPriority>`, the item with the **lowest priority value** is removed first. A smaller number therefore means earlier processing in this example. This is a convention you must define for the application, not an automatic interpretation of what "high priority" means.[^priority-queue]

Each insertion supplies two pieces of information: the element to store and its priority. A routine request arrives first, but the urgent request leaves first because its priority value is smaller.


In [6]:
using System;
using System.Collections.Generic;

PriorityQueue<string, int> requests = new PriorityQueue<string, int>();
requests.Enqueue("Routine restock", 3);
requests.Enqueue("Urgent replacement", 1);
requests.Enqueue("Scheduled pickup", 2);

while (requests.Count > 0)
{
    Console.WriteLine(requests.Dequeue());
}


Urgent replacement
Scheduled pickup
Routine restock


The removed items follow the priority rule even though they arrived in another order. A priority queue does not guarantee FIFO order among items with equal priority. If arrival order must break ties, the application needs an explicit tie-breaking rule. The unique priorities in this example avoid that ambiguity. Chapter 18 develops priorities and heap implementations.

**Check Your Understanding:** Which collection fits each task: undo the latest edit, serve the earliest waiting request, or process the smallest urgency number first?

```{dropdown} Suggested Answer
Use a stack for the latest edit, a queue for arrival-order requests, and a priority queue for the smallest urgency value. All three may store the same element type; the difference is the retrieval rule.
```


## Traversal and Shared References

```{index} traversal; enumeration; IEnumerable; reference; alias; mutation
```


**Traversal**, also called **enumeration**, means visiting a collection's elements. `foreach` provides familiar syntax for this work, but the order and the kind of element you receive depend on the collection. Arrays and lists follow their sequence order. A dictionary enumerates key/value entries. A set supports enumeration, but its purpose is membership rather than sorted traversal. `PriorityQueue` is a special case: it exposes an `UnorderedItems` view for enumeration instead of supporting `foreach` directly on the queue. That view does not promise priority order; use successive removals when you need that order.

The enumerable collection types demonstrated here implement `IEnumerable<T>`, an interface describing how to enumerate values.[^collection-interface] You do not need to write an interface to use these library collections. Later chapters explain how a type can provide this capability itself. Also distinguish traversal from removal: looping through a queue with `foreach` does not serve and remove its waiting items.

Here a traversal computes a total without changing the stored quantities. The accumulator starts at zero, so an empty list would produce a total of zero.


In [8]:
using System;
using System.Collections.Generic;

List<int> quantities = new List<int> { 2, 1, 3 };
int total = 0;
foreach (int quantity in quantities)
{
    total += quantity;
}

Console.WriteLine($"Entries: {quantities.Count}");
Console.WriteLine($"Total units: {total}");


Entries: 3
Total units: 6


The mutable library collection classes shown here are **reference types**. A variable refers to a collection object. Assigning that variable to another variable copies the reference, rather than making a second collection. Two names for the same object are **aliases**. Arrays have reference behavior too; this continues the idea introduced in Chapter 7.

Consider a basket shown in two parts of a program. If both variables refer to the same basket, adding an item through one variable is visible through the other. If you need a separate snapshot of a list's elements, you must explicitly create a new collection.


In [9]:
using System;
using System.Collections.Generic;

List<string> basket = new List<string> { "NOTEBOOK" };
List<string> sameBasket = basket;
List<string> snapshot = new List<string>(basket);

sameBasket.Add("PEN");

Console.WriteLine($"Original basket: {basket.Count}");
Console.WriteLine($"Alias: {sameBasket.Count}");
Console.WriteLine($"Snapshot: {snapshot.Count}");


Original basket: 2
Alias: 2
Snapshot: 1


`basket` and `sameBasket` refer to one list containing two strings. `snapshot` is a different list containing the one string that was present when it was created. This is a copy of the elements, not a general promise to duplicate every object an element might refer to. Chapter 11 revisits that distinction with objects and composition.

Mutation means changing an existing object's state, such as adding to a list. Reassigning a variable to a different list is a different action. This distinction matters when collections are passed to methods or shared between parts of a program. An operation can affect more code than the variable name immediately in front of you suggests.

You should also check whether an operation is valid during traversal. In particular, adding or removing list elements inside a `foreach` over that same list invalidates its enumeration. Section 8.2 shows a safe removal pattern. A short loop can still be incorrect if it ignores the collection's operating rules.


**Interactive Exercise: An Order Summary**

A small order contains quantities `2`, `1`, and `3`. The starter stores them in a `List<int>`. Compute the total with `foreach` and store that total under the key `"ORD-101"` in a `Dictionary<string, int>`. Print the entry count and the saved total. Expected output is `Line items: 3` and `Saved total: 6` on separate lines.

After running your code, explain why the quantities form a list and why the saved summary uses a dictionary. Change the list to be empty and predict both results before running it again.


In [10]:
// Exercise: An Order Summary
using System;
using System.Collections.Generic;

List<int> quantities = new List<int> { 2, 1, 3 };

// TODO: Calculate the total, save it by order code, and print the results.
// Your code starts here.



// Your code ends here.


In [11]:
// Solution
using System;
using System.Collections.Generic;

List<int> quantities = new List<int> { 2, 1, 3 };
int total = 0;
foreach (int quantity in quantities)
{
    total += quantity;
}
Dictionary<string, int> orderTotals = new Dictionary<string, int>();
orderTotals["ORD-101"] = total;

Console.WriteLine($"Line items: {quantities.Count}");
Console.WriteLine($"Saved total: {orderTotals["ORD-101"]}");


Line items: 3
Saved total: 6


The list stores a sequence of quantities, including any repeated quantities. The dictionary associates an order's identity with its total. With an empty list, the entry count and calculated total are both zero. The dictionary still has one entry because the program saves that zero total under `ORD-101`; no line items is different from no saved summary.


## Summary



Collections combine stored values with rules for using them. Arrays and lists provide indexed sequences; linked lists organize nodes; dictionaries support key-based lookup; sets emphasize uniqueness and membership; stacks, queues, and priority queues determine what comes next. Generic type arguments specify the types of stored elements, keys, values, or priorities.

Choose the collection by explaining the task's access pattern, ordering, duplicates, and changes in size. Then check the details of its contract. The following sections put that reasoning into practice with lists and dictionaries.


```{rubric} Footnotes
```

[^collection-guide]: Microsoft. [*Collections and Data Structures*](https://learn.microsoft.com/en-us/dotnet/standard/collections/) and [*Selecting a Collection Class*](https://learn.microsoft.com/en-us/dotnet/standard/collections/selecting-a-collection-class). Microsoft Learn. Accessed October 16, 2026. The table adapts the documentation's operation-based selection approach; its eight-family grouping and bookstore examples are the author's teaching organization. The priority-queue entry is additionally supported by the API reference below.
[^linked-list]: The .NET [LinkedList<T> reference](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.linkedlist-1?view=net-10.0) describes its node-based, doubly linked organization.
[^priority-queue]: The .NET [PriorityQueue<TElement,TPriority> reference](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2?view=net-10.0) specifies lowest-priority removal and does not guarantee FIFO behavior for equal priorities.
[^collection-interface]: `IEnumerable<T>` supplies an enumerator used by `foreach`. Enumeration order belongs to each type's contract; supporting enumeration does not imply sorted, insertion, or priority order.

[^collection-operations]: Microsoft Learn, .NET 10 API reference: [List<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.list-1?view=net-10.0), [LinkedList<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.linkedlist-1?view=net-10.0), [Dictionary<TKey,TValue>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.dictionary-2?view=net-10.0), [HashSet<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.hashset-1?view=net-10.0), [Stack<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.stack-1?view=net-10.0), [Queue<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.queue-1?view=net-10.0), and [PriorityQueue<TElement,TPriority>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2?view=net-10.0). This table compares selected operations rather than listing each type's complete API.
